In [1]:
%load_ext autotime

from ast import literal_eval
import json
from pathlib import Path
import re
import time
from tqdm.auto import tqdm

import pandas as pd

from data_snapshot.utils import load_json
from field_labeling import (
    load_prompt,
    analyze_field_profile,
    process_response,
    make_classification_model,
)

# Inputs

In [2]:
# Inputs
SYSTEM_PROMPT_PATH = "prompts/field_labeling_using_schema_system.md"
USER_PROMPT_PATH = "prompts/field_labeling_using_schema_user.md"
SCHEMA_REFERENCE_PATH = "outputs/schema_reference_v1.1.md"
MODEL_NAME = "gpt-5.4-mini"
SLEEP_SECONDS = 0.2  # safety throttle
OUTPUT_PATH = f"data/field_labeling_using_schema_results.jsonl"
PROFILES_PATH = "outputs/3.0-field_profiles.csv"

# Load data

In [3]:
system_prompt = load_prompt(SYSTEM_PROMPT_PATH)
user_prompt_template = load_prompt(USER_PROMPT_PATH)
schema_ref = load_prompt(SCHEMA_REFERENCE_PATH)
df_profiles = pd.read_csv(PROFILES_PATH)

# Prepare prompts

In [4]:
user_prompt_template = user_prompt_template.replace(
    "{{SCHEMA_MARKDOWN}}", schema_ref.split("\n", 2)[-1]
)

# Main pipeline

## Create ClassificationModel

In [5]:
import re

# Regex matches lines starting with optional whitespace, exactly two hashes, and a space
h2_pattern = re.compile(r"^\s*##\s+(.+)$")
h2_headers = []

with open(SCHEMA_REFERENCE_PATH, "r", encoding="utf-8") as file:
    for line in file:
        match = h2_pattern.match(line)
        if match:
            # Extract only the header text captured inside the parentheses
            h2_headers.append(match.group(1).strip())

len(h2_headers)

36

In [6]:
canonical_names = tuple(h2_headers)
ClassificationModel = make_classification_model(canonical_names + ("not_in_schema",))

## Create skip list

In [7]:
# Build skip list
to_skip = set()
if Path(OUTPUT_PATH).exists():
    with open(OUTPUT_PATH, "r", encoding="utf-8") as f:
        for line in f:
            to_skip.add(json.loads(line)["metadata_field"])

len(to_skip)

0

## Smoke test

In [8]:
# # Smoke test
# df_profiles = df_profiles.sample(1)
# df_profiles = df_profiles.head(1)
# df_profiles = df_profiles[df_profiles["metadata_field"] == "missing_value_marker"].copy()

## Main loop

In [9]:
# Initialize dir
Path(OUTPUT_PATH).parent.mkdir(parents=True, exist_ok=True)

# Main loop
with open(OUTPUT_PATH, "a", encoding="utf-8") as out_f:
    for _, field_row in tqdm(df_profiles.iterrows(), total=len(df_profiles)):
        metadata_field = field_row["metadata_field"]
        
        # Skip if in skip list
        if metadata_field in to_skip:
            print(f"Skipped: {metadata_field}")
            continue

        # Base row — every row has the same keys
        row = {
            "metadata_field": metadata_field,
            "model": MODEL_NAME,
            "raw_output": None,
            "usage": None,
            "cost": None,
            "status": None,
            "incomplete_details": None,
            "parsed_output": None,
            "error": None,
        }

        try:
            # Create user prompt from template
            user_prompt = user_prompt_template
            top_description_values_str = "\n".join(["- " + x.strip() for x in literal_eval(field_row["top_description_values"])])
            try:
                top_observed_values_str = "\n".join(["- " + x.strip() for x in literal_eval(field_row["top_observed_values"])])
            except ValueError:
                top_observed_values_str = "N/A"
            replace_dict = {
                "{{metadata_field}}": metadata_field,
                "{{top_description_values}}": top_description_values_str,
                "{{top_observed_values}}": top_observed_values_str,
            }
            for old, new in replace_dict.items():
                user_prompt = user_prompt.replace(old, new)
        
            # Responses API
            response = analyze_field_profile(
                system_prompt=system_prompt,
                user_prompt=user_prompt,
                model=MODEL_NAME,
                classification_model=ClassificationModel,
            )

            # Parse and process response
            result = process_response(response, model=MODEL_NAME)

            # Log data
            row["raw_output"] = result["raw_output"]
            row["usage"] = result["usage"]
            row["cost"] = result["cost"]
            row["status"] = result["status"]
            row["incomplete_details"] = result["incomplete_details"]
            row["parsed_output"] = result["parsed_output"]
            row["error"] = result["error"]  # None on success, message on parse failure

        except Exception as e:
            row["error"] = str(e)

        # Single write point — always executes
        out_f.write(json.dumps(row, ensure_ascii=False) + "\n")
        out_f.flush()

        time.sleep(SLEEP_SECONDS)

  0%|          | 0/833 [00:00<?, ?it/s]